# 1日目終了 vs 継続試合 分析（2日目限定で明確化する特徴量の抽出）

このノートブックは cross-dataset ノートブックの前処理方針を参考に、以下を分析します。
- 1日目で終わる試合と終わらない試合の差
- 2日目（day=1）に限定したときに識別性が上がる特徴量

In [12]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import f_classif

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
CONFIG_PATH = PROJECT_ROOT / "config" / "training_config.json"
OUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "day_transition_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from src.pipelines.training_pipeline import load_training_config, get_data_paths

print(pd.DataFrame([
    {"key": "PROJECT_ROOT", "value": str(PROJECT_ROOT)},
    {"key": "CONFIG_PATH", "value": str(CONFIG_PATH)},
    {"key": "OUT_DIR", "value": str(OUT_DIR)},
]))

            key                                              value
0  PROJECT_ROOT  c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定...
1   CONFIG_PATH  c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定...
2       OUT_DIR  c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定...


In [13]:
RUN_OPTIONS = {
    "target_day_for_2day": 1,
    "top_k_features": 30,
}

config = load_training_config(CONFIG_PATH)
data_paths = get_data_paths(config)
if len(data_paths) == 0:
    raise ValueError("No data_paths found in training_config.json")

frames = []
for p in data_paths:
    p_path = Path(p)
    df_raw = pd.read_csv(p)
    if "day" not in df_raw.columns:
        print(f"skip {p_path.name}: no day column")
        continue
    df_raw["dataset_tag"] = p_path.stem
    frames.append(df_raw)

if not frames:
    raise ValueError("No valid data loaded.")

df_all = pd.concat(frames, ignore_index=True)
print(df_all.shape)
display(df_all[["dataset_tag", "day"]].groupby("dataset_tag").agg(["count", "min", "max"]))

✓ Data files selected:
  - all_feature_table_2025sp17_with_talks.csv
  - all_feature_table_2025summer_with_talks2.csv
(2250, 232)


day        
                                         count min max
dataset_tag                                           
all_feature_table_2025sp17_with_talks     1060   1   2
all_feature_table_2025summer_with_talks2  1190   1   2

In [7]:
game_summary = (
    df_all.groupby(["dataset_tag", "source_file"], as_index=False)
    .agg(max_day=("day", "max"), n_rows=("day", "size"))
)
game_summary["reach_day2"] = (game_summary["max_day"] >= 2).astype(int)
game_summary["game_type"] = np.where(game_summary["reach_day2"] == 1, "continue_to_day2", "end_day1")

display(game_summary.head())
display(
    game_summary.groupby(["dataset_tag", "game_type"], as_index=False)
    .size()
    .rename(columns={"size": "games"})
)

,dataset_tag,source_file,max_day,n_rows,reach_day2,game_type
0,all_feature_table_2025sp17_with_talks,taged_1746429865_osawalab_ai168wolf_CamelliaDr...,2,10,1,continue_to_day2
1,all_feature_table_2025sp17_with_talks,taged_1746431268_mille_sunamelli_kanolab_Camel...,2,10,1,continue_to_day2
2,all_feature_table_2025sp17_with_talks,taged_1746431630_kanolab_mille_ai168wolf_osawa...,2,10,1,continue_to_day2
3,all_feature_table_2025sp17_with_talks,taged_1746432378_mille_ai168wolf_osawalab_Came...,2,10,1,continue_to_day2
4,all_feature_table_2025sp17_with_talks,taged_1746433451_kanolab_ai168wolf_sunamelli_o...,2,10,1,continue_to_day2


,dataset_tag,game_type,games
0,all_feature_table_2025sp17_with_talks,continue_to_day2,106
1,all_feature_table_2025summer_with_talks2,continue_to_day2,119


In [23]:
# 比較戦略：このデータセットにはday=1で終わる試合がないため、day1とday2の特徴量の変化を比較
# これは「2日目データがあると何が変わるか」を示す
print("=" * 70)
print("分析戦略: Day-1 vs Day-2 特徴量比較")
print("=" * 70)
print(f"Data contains {(df_all['day'] == 1).sum()} day=1 rows")
print(f"Data contains {(df_all['day'] == 2).sum()} day=2 rows")

# day 1とday 2のデータを準備
df_day1 = df_all[df_all["day"] == 1].copy()
df_day2 = df_all[df_all["day"] == 2].copy()

if df_day1.empty or df_day2.empty:
    raise ValueError("Cannot compare: missing day=1 or day=2 data.")

# 共通の列を取得
base_drop_cols = [
    "id", "role", "source_file", "dataset_tag", "day", "role_encoded",
    "Est_id_Fact_role", "Est_id_Est_roles", "character_name",
    "agent_name", "combined_text", "seer_co_order"
]
meta_raw_cols = [
    "True_Div_result_1", "True_Div_recepient_id_1",
    "True_Div_result_2", "True_Div_recepient_id_2",
    "exec_id", "attack_id"
]
leakage_drop_cols = config.get("leakage_drop_columns", [])

# IDリーク対策: Div_recepient_id本体と派生列 + "_id"で終わる列を除外
id_sensitive_cols = [
    c for c in df_all.columns
    if c == "Div_recepient_id"
    or c.startswith("Div_recepient_id_")
    or c == "Div_recipient_id"
    or c.startswith("Div_recipient_id_")
    or c.endswith("_id")
]

drop_cols = list(set(base_drop_cols + meta_raw_cols + leakage_drop_cols + id_sensitive_cols))
print(f"Excluded id-sensitive columns: {len(id_sensitive_cols)}")

# 両日のデータで特徴量を抽出
feature_df_day1 = df_day1.drop(columns=drop_cols, errors="ignore")
feature_df_day1 = feature_df_day1.apply(pd.to_numeric, errors="coerce")

feature_df_day2 = df_day2.drop(columns=drop_cols, errors="ignore")
feature_df_day2 = feature_df_day2.apply(pd.to_numeric, errors="coerce")

# 両方に共通の列のみ使用
common_cols = list(set(feature_df_day1.columns) & set(feature_df_day2.columns))
numeric_cols = [c for c in common_cols if feature_df_day1[c].notna().sum() > 0 and feature_df_day2[c].notna().sum() > 0]

print(f"\nCommon numeric feature columns: {len(numeric_cols)}")

# Role情報
day1_roles = df_day1["role"].values
day2_roles = df_day2["role"].values

rows = []
for role_val in sorted(set(day1_roles) | set(day2_roles)):
    day1_mask = day1_roles == role_val
    day2_mask = day2_roles == role_val

    n_day1 = day1_mask.sum()
    n_day2 = day2_mask.sum()

    if n_day1 < 5 or n_day2 < 5:
        continue

    for col in numeric_cols:
        s1 = feature_df_day1.loc[day1_mask, col].dropna()
        s2 = feature_df_day2.loc[day2_mask, col].dropna()

        if len(s1) < 5 or len(s2) < 5:
            continue

        m1, m2 = float(s1.mean()), float(s2.mean())
        v1, v2 = float(s1.var(ddof=1)), float(s2.var(ddof=1))
        pooled = np.sqrt((v1 + v2) / 2.0) if (v1 + v2) > 0 else np.nan
        smd = (m2 - m1) / pooled if pooled and np.isfinite(pooled) and pooled > 0 else np.nan

        rows.append({
            "role": role_val,
            "feature": col,
            "mean_day1": m1,
            "mean_day2": m2,
            "std_mean_diff": smd,
            "n_day1": int(len(s1)),
            "n_day2": int(len(s2)),
        })

day_transition_df = pd.DataFrame(rows)
if not day_transition_df.empty:
    day_transition_df["abs_std_mean_diff"] = day_transition_df["std_mean_diff"].abs()
    print(f"\nAnalyzed {len(day_transition_df)} feature-role combinations.")
    print("\nTop features that change from day=1 to day=2 (by absolute standardized mean difference):\n")

    top_features = day_transition_df.sort_values("abs_std_mean_diff", ascending=False).head(RUN_OPTIONS["top_k_features"])
    display(top_features)
else:
    print("No feature-role pairs found for analysis (sample sizes too small).")
    day_transition_df = pd.DataFrame()

分析戦略: Day-1 vs Day-2 特徴量比較
Data contains 1125 day=1 rows
Data contains 1125 day=2 rows
Excluded id-sensitive columns: 31

Common numeric feature columns: 147

Analyzed 588 feature-role combinations.

Top features that change from day=1 to day=2 (by absolute standardized mean difference):



,role,feature,mean_day1,mean_day2,std_mean_diff,n_day1,n_day2,abs_std_mean_diff
220,SEER,Fact_recepient_id_Sus,2.000000,0.555556,-1.134085,31,9,1.134085
213,SEER,Fact_recepient_id_ReqDiscuss,4.129032,1.111111,-1.100990,31,9,1.100990
507,WEREWOLF,Fact_recepient_id_ReqDiscuss,3.526316,1.000000,-1.060649,19,17,1.060649
126,POSSESSED,day1_vote_id_ReqDiscuss,3.100000,0.475000,-0.939678,200,200,0.939678
66,POSSESSED,Fact_recepient_id_ReqDiscuss,3.050000,0.900000,-0.904664,20,10,0.904664
90,POSSESSED,Fact_recepient_id_calm,3.200000,0.400000,-0.901611,20,10,0.901611
345,VILLAGER,Fact_recepient_id_difficult,0.380952,0.000000,-0.864996,42,20,0.864996
78,POSSESSED,Fact_recepient_id_ReqListen,0.600000,0.000000,-0.853030,20,10,0.853030
360,VILLAGER,Fact_recepient_id_ReqDiscuss,3.142857,0.800000,-0.840938,42,20,0.840938
420,VILLAGER,day1_vote_id_ReqDiscuss,2.645309,0.423341,-0.832713,437,437,0.832713


In [24]:
# 役職分離性の向上を測定（Day-1限定 vs 全日データ）
# ANOVA F値でどの特徴量がday1に限定すると役職分類に有効になるかを測定
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import LabelEncoder

print("=" * 70)
print("役職分離性分析: Day-1限定 vs 全日データ")
print("=" * 70)

# 全日データ
df_role_all = df_all.copy()
# Day-1限定
df_role_day1 = df_all[df_all["day"] == 1].copy()

if df_role_day1.empty:
    raise ValueError("No day=1 rows available.")

base_drop_cols_role = [
    "id", "role", "source_file", "dataset_tag", "day", "role_encoded",
    "Est_id_Fact_role", "Est_id_Est_roles", "character_name",
    "agent_name", "combined_text", "seer_co_order"
]
meta_raw_cols_role = [
    "True_Div_result_1", "True_Div_recepient_id_1",
    "True_Div_result_2", "True_Div_recepient_id_2",
    "exec_id", "attack_id"
]

# IDリーク対策: Div_recepient_id本体と派生列 + "_id"で終わる列を除外
id_sensitive_cols_role = [
    c for c in df_role_all.columns
    if c == "Div_recepient_id"
    or c.startswith("Div_recepient_id_")
    or c == "Div_recipient_id"
    or c.startswith("Div_recipient_id_")
    or c.endswith("_id")
]

drop_cols_role = list(set(base_drop_cols_role + meta_raw_cols_role + config.get("leakage_drop_columns", []) + id_sensitive_cols_role))
print(f"Excluded id-sensitive columns: {len(id_sensitive_cols_role)}")

# ラベル準備
role_encoder = LabelEncoder()
y_all = role_encoder.fit_transform(df_role_all["role"].astype(str))
y_day1 = role_encoder.transform(df_role_day1["role"].astype(str))

candidate_features = [c for c in df_role_all.columns if c not in drop_cols_role]

# 特徴量行列の準備
X_all = df_role_all[candidate_features].apply(pd.to_numeric, errors="coerce")
X_day1 = df_role_day1[candidate_features].apply(pd.to_numeric, errors="coerce")

# 有効な特徴量のフィルタリング
valid_features = []
for c in candidate_features:
    if X_all[c].notna().sum() < 20:
        continue
    if X_day1[c].notna().sum() < 20:
        continue
    valid_features.append(c)

X_all = X_all[valid_features].fillna(0)
X_day1 = X_day1[valid_features].fillna(0)

print(f"\nValid features for ANOVA: {len(valid_features)}")
print(f"Sample sizes - All days: {len(X_all)}, Day-1 only: {len(X_day1)}")

# ANOVA F値計算
f_all, _ = f_classif(X_all, y_all)
f_day1, _ = f_classif(X_day1, y_day1)

# 結果をDataFrameにまとめる
role_sep_df = pd.DataFrame({
    "feature": valid_features,
    "f_score_all_days": f_all,
    "f_score_day1_only": f_day1,
})
role_sep_df["f_score_delta_day1_minus_all"] = role_sep_df["f_score_day1_only"] - role_sep_df["f_score_all_days"]
role_sep_df["f_score_ratio_day1_over_all"] = (role_sep_df["f_score_day1_only"] + 1e-9) / (role_sep_df["f_score_all_days"] + 1e-9)
role_sep_df = role_sep_df.sort_values("f_score_delta_day1_minus_all", ascending=False).reset_index(drop=True)

print(f"\nFeatures showing highest improvement in day-1-limited model:\n")
display(role_sep_df.head(RUN_OPTIONS["top_k_features"]))

役職分離性分析: Day-1限定 vs 全日データ
Excluded id-sensitive columns: 31

Valid features for ANOVA: 148
Sample sizes - All days: 2250, Day-1 only: 1125

Features showing highest improvement in day-1-limited model:



,feature,f_score_all_days,f_score_day1_only,f_score_delta_day1_minus_all,f_score_ratio_day1_over_all
0,Est_id_Req(V),1.014707,4.012787,2.998080,3.954627
1,Agr_id_Req(CO),0.530271,2.681016,2.150746,5.055938
2,Req(V),8.302878,10.212292,1.909414,1.229970
3,Fact_recepient_id_IF,0.638584,1.761392,1.122808,2.758279
4,Agr_id_confused,0.072994,0.947057,0.874063,12.974364
5,Vote_recepient_id_ReqListen,0.421863,1.287546,0.865683,3.052049
6,Fact,0.300521,1.101456,0.800936,3.665161
7,Est_id_Req(T),1.072580,1.809335,0.736755,1.686900
8,Fact_recepient_id_Mt,0.377939,1.048278,0.670339,2.773672
9,Est_id_Sus,1.915084,2.556408,0.641325,1.334881


In [28]:
# 大会ごと（dataset_tagごと）に、上記と同じ比較を実施
# 1) Day-1 vs Day-2 特徴量変化
# 2) 役職分離性（Day-1限定 vs 全日）

from sklearn.feature_selection import f_classif
from sklearn.preprocessing import LabelEncoder

print("=" * 70)
print("大会別分析: Day-1 vs Day-2 / 役職分離性")
print("=" * 70)

tournament_transition_results = []
tournament_role_sep_results = []

for dataset_tag in sorted(df_all["dataset_tag"].dropna().unique()):
    df_t = df_all[df_all["dataset_tag"] == dataset_tag].copy()
    print(f"\n[dataset] {dataset_tag}  rows={len(df_t)}")

    # --- 共通除外列（既存ロジックを踏襲） ---
    base_drop_cols_t = [
        "id", "role", "source_file", "dataset_tag", "day", "role_encoded",
        "Est_id_Fact_role", "Est_id_Est_roles", "character_name",
        "agent_name", "combined_text", "seer_co_order"
    ]
    meta_raw_cols_t = [
        "True_Div_result_1", "True_Div_recepient_id_1",
        "True_Div_result_2", "True_Div_recepient_id_2",
        "exec_id", "attack_id"
    ]
    leakage_drop_cols_t = config.get("leakage_drop_columns", [])

    id_sensitive_cols_t = [
        c for c in df_t.columns
        if c == "Div_recepient_id"
        or c.startswith("Div_recepient_id_")
        or c == "Div_recipient_id"
        or c.startswith("Div_recipient_id_")
        or c.endswith("_id")
    ]

    drop_cols_t = list(set(base_drop_cols_t + meta_raw_cols_t + leakage_drop_cols_t + id_sensitive_cols_t))

    # =========================
    # 1) Day-1 vs Day-2 比較
    # =========================
    df_t_day1 = df_t[df_t["day"] == 1].copy()
    df_t_day2 = df_t[df_t["day"] == 2].copy()

    if not df_t_day1.empty and not df_t_day2.empty:
        X_day1_t = df_t_day1.drop(columns=drop_cols_t, errors="ignore").apply(pd.to_numeric, errors="coerce")
        X_day2_t = df_t_day2.drop(columns=drop_cols_t, errors="ignore").apply(pd.to_numeric, errors="coerce")

        common_cols_t = list(set(X_day1_t.columns) & set(X_day2_t.columns))
        numeric_cols_t = [
            c for c in common_cols_t
            if X_day1_t[c].notna().sum() > 0 and X_day2_t[c].notna().sum() > 0
        ]

        roles_day1_t = df_t_day1["role"].values
        roles_day2_t = df_t_day2["role"].values

        rows_t = []
        for role_val in sorted(set(roles_day1_t) | set(roles_day2_t)):
            m1 = roles_day1_t == role_val
            m2 = roles_day2_t == role_val
            if m1.sum() < 5 or m2.sum() < 5:
                continue

            for col in numeric_cols_t:
                s1 = X_day1_t.loc[m1, col].dropna()
                s2 = X_day2_t.loc[m2, col].dropna()
                if len(s1) < 5 or len(s2) < 5:
                    continue

                mean1, mean2 = float(s1.mean()), float(s2.mean())
                var1, var2 = float(s1.var(ddof=1)), float(s2.var(ddof=1))
                pooled = np.sqrt((var1 + var2) / 2.0) if (var1 + var2) > 0 else np.nan
                smd = (mean2 - mean1) / pooled if pooled and np.isfinite(pooled) and pooled > 0 else np.nan

                rows_t.append({
                    "dataset_tag": dataset_tag,
                    "role": role_val,
                    "feature": col,
                    "mean_day1": mean1,
                    "mean_day2": mean2,
                    "std_mean_diff": smd,
                    "n_day1": int(len(s1)),
                    "n_day2": int(len(s2)),
                })

        if rows_t:
            df_trans_t = pd.DataFrame(rows_t)
            df_trans_t["abs_std_mean_diff"] = df_trans_t["std_mean_diff"].abs()
            tournament_transition_results.append(df_trans_t)
            print(f"  transition rows={len(df_trans_t)}")
        else:
            print("  transition rows=0 (sample/feature不足)")
    else:
        print("  transition skipped (day1/day2片方が空)")

    # ====================================
    # 2) 役職分離性（Day-1限定 vs 全日）
    # ====================================
    df_role_all_t = df_t.copy()
    df_role_day1_t = df_t[df_t["day"] == 1].copy()

    if not df_role_day1_t.empty and df_role_all_t["role"].nunique() >= 2:
        role_encoder_t = LabelEncoder()
        y_all_t = role_encoder_t.fit_transform(df_role_all_t["role"].astype(str))
        y_day1_t = role_encoder_t.transform(df_role_day1_t["role"].astype(str))

        candidate_t = [c for c in df_role_all_t.columns if c not in drop_cols_t]
        X_all_t = df_role_all_t[candidate_t].apply(pd.to_numeric, errors="coerce")
        X_day1_t2 = df_role_day1_t[candidate_t].apply(pd.to_numeric, errors="coerce")

        valid_t = []
        for c in candidate_t:
            if X_all_t[c].notna().sum() < 20:
                continue
            if X_day1_t2[c].notna().sum() < 20:
                continue
            valid_t.append(c)

        if valid_t:
            Xa = X_all_t[valid_t].fillna(0)
            Xd = X_day1_t2[valid_t].fillna(0)
            f_all_t, _ = f_classif(Xa, y_all_t)
            f_day1_t, _ = f_classif(Xd, y_day1_t)

            df_role_t = pd.DataFrame({
                "dataset_tag": dataset_tag,
                "feature": valid_t,
                "f_score_all_days": f_all_t,
                "f_score_day1_only": f_day1_t,
            })
            df_role_t["f_score_delta_day1_minus_all"] = df_role_t["f_score_day1_only"] - df_role_t["f_score_all_days"]
            df_role_t["f_score_ratio_day1_over_all"] = (df_role_t["f_score_day1_only"] + 1e-9) / (df_role_t["f_score_all_days"] + 1e-9)

            tournament_role_sep_results.append(df_role_t)
            print(f"  role-sep rows={len(df_role_t)}")
        else:
            print("  role-sep rows=0 (valid featureなし)")
    else:
        print("  role-sep skipped (day1空 or role種類不足)")

# 連結結果
if tournament_transition_results:
    day_transition_by_tournament_df = pd.concat(tournament_transition_results, ignore_index=True)
    day_transition_by_tournament_df = day_transition_by_tournament_df.sort_values(
        ["dataset_tag", "abs_std_mean_diff"], ascending=[True, False]
    ).reset_index(drop=True)
else:
    day_transition_by_tournament_df = pd.DataFrame()

if tournament_role_sep_results:
    role_sep_by_tournament_df = pd.concat(tournament_role_sep_results, ignore_index=True)
    role_sep_by_tournament_df = role_sep_by_tournament_df.sort_values(
        ["dataset_tag", "f_score_delta_day1_minus_all"], ascending=[True, False]
    ).reset_index(drop=True)
else:
    role_sep_by_tournament_df = pd.DataFrame()

print("\n" + "=" * 70)
print("大会別分析サマリ")
print("=" * 70)
print(f"day_transition_by_tournament_df: {day_transition_by_tournament_df.shape}")
print(f"role_sep_by_tournament_df: {role_sep_by_tournament_df.shape}")

if not day_transition_by_tournament_df.empty:
    print("\nDay-1 vs Day-2: 各大会の上位特徴（先頭10件）")
    display(day_transition_by_tournament_df.groupby("dataset_tag", as_index=False).head(10))

if not role_sep_by_tournament_df.empty:
    print("\n役職分離性: 各大会の上位特徴（先頭10件）")
    display(role_sep_by_tournament_df.groupby("dataset_tag", as_index=False).head(10))

大会別分析: Day-1 vs Day-2 / 役職分離性

[dataset] all_feature_table_2025sp17_with_talks  rows=1060
  transition rows=552
  role-sep rows=148

[dataset] all_feature_table_2025summer_with_talks2  rows=1190
  transition rows=534
  role-sep rows=148

大会別分析サマリ
day_transition_by_tournament_df: (1086, 9)
role_sep_by_tournament_df: (296, 6)

Day-1 vs Day-2: 各大会の上位特徴（先頭10件）


,dataset_tag,role,feature,mean_day1,mean_day2,std_mean_diff,n_day1,n_day2,abs_std_mean_diff
0,all_feature_table_2025sp17_with_talks,WEREWOLF,Fact_recepient_id_ReqDiscuss,3.866667,0.714286,-1.270425,15,7,1.270425
1,all_feature_table_2025sp17_with_talks,POSSESSED,day1_vote_id_ReqDiscuss,4.836538,0.701923,-1.261764,104,104,1.261764
2,all_feature_table_2025sp17_with_talks,VILLAGER,day1_vote_id_ReqDiscuss,4.568720,0.715640,-1.209174,211,211,1.209174
3,all_feature_table_2025sp17_with_talks,SEER,ReqDiscuss,4.207547,0.433962,-1.115435,106,106,1.115435
4,all_feature_table_2025sp17_with_talks,VILLAGER,Vote_recepient_id_ReqDiscuss,4.870968,2.157895,-1.080398,31,19,1.080398
5,all_feature_table_2025sp17_with_talks,POSSESSED,ReqDiscuss,4.320755,0.801887,-1.054621,106,106,1.054621
6,all_feature_table_2025sp17_with_talks,SEER,Sus_recepient_id_ReqDiscuss,4.328767,1.285714,-1.051883,73,14,1.051883
7,all_feature_table_2025sp17_with_talks,POSSESSED,Vote_recepient_id_confused,0.125000,0.600000,1.043855,24,5,1.043855
8,all_feature_table_2025sp17_with_talks,VILLAGER,Fact_recepient_id_difficult,0.483871,0.000000,-1.011070,31,9,1.011070
9,all_feature_table_2025sp17_with_talks,VILLAGER,day1_vote_id_Sus,1.545024,0.308057,-0.975517,211,211,0.975517



役職分離性: 各大会の上位特徴（先頭10件）


,dataset_tag,feature,f_score_all_days,f_score_day1_only,f_score_delta_day1_minus_all,f_score_ratio_day1_over_all
0,all_feature_table_2025sp17_with_talks,Agr_id_contradiction,0.891856,2.050449,1.158593,2.299081
1,all_feature_table_2025sp17_with_talks,Fact_recepient_id_IF,0.721545,1.774458,1.052913,2.459248
2,all_feature_table_2025sp17_with_talks,Agr_id_confused,0.098273,1.087339,0.989065,11.064463
3,all_feature_table_2025sp17_with_talks,Sus_recepient_id_difficult,0.554424,1.485086,0.930663,2.678613
4,all_feature_table_2025sp17_with_talks,Vote,1.544699,2.361335,0.816636,1.528670
5,all_feature_table_2025sp17_with_talks,Dis_id_Req(CO),0.594406,1.346129,0.751723,2.264662
6,all_feature_table_2025sp17_with_talks,Est_id_Vote,0.279335,0.955207,0.675872,3.419578
7,all_feature_table_2025sp17_with_talks,Est_id_calm,0.661192,1.306431,0.645239,1.975873
8,all_feature_table_2025sp17_with_talks,Est_id_contradiction,0.943942,1.555187,0.611246,1.647546
9,all_feature_table_2025sp17_with_talks,Vote_recepient_id_Mt,0.418145,0.995928,0.577784,2.381780


In [27]:
# 大会別分析結果をCSV保存
OUT_ANALYSIS_DIR = OUT_DIR / "day_transition_analysis"
OUT_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

day_transition_by_tournament_path = OUT_ANALYSIS_DIR / "day1_vs_day2_feature_changes_by_tournament.csv"
role_sep_by_tournament_path = OUT_ANALYSIS_DIR / "day1_role_separation_improvement_by_tournament.csv"

if 'day_transition_by_tournament_df' in dir() and not day_transition_by_tournament_df.empty:
    day_transition_by_tournament_df.to_csv(day_transition_by_tournament_path, index=False)
    print(f"Saved: {day_transition_by_tournament_path}")
    print(f"  shape={day_transition_by_tournament_df.shape}")
else:
    print("No tournament transition result to save.")

if 'role_sep_by_tournament_df' in dir() and not role_sep_by_tournament_df.empty:
    role_sep_by_tournament_df.to_csv(role_sep_by_tournament_path, index=False)
    print(f"Saved: {role_sep_by_tournament_path}")
    print(f"  shape={role_sep_by_tournament_df.shape}")
else:
    print("No tournament role-separation result to save.")

Saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\day_transition_analysis\day_transition_analysis\day1_vs_day2_feature_changes_by_tournament.csv
  shape=(1086, 9)
Saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\day_transition_analysis\day_transition_analysis\day1_role_separation_improvement_by_tournament.csv
  shape=(296, 6)
